In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

sales = pd.read_csv('kc_house_data.csv')


train_data, test_data = train_test_split(sales, test_size=0.8, random_state=0)

def get_numpy_data(dataframe, features, output):
    df_temp = dataframe.copy()
    
    df_temp['constant'] = 1.0
    
    all_features = ['constant'] + features
    
    feature_matrix = df_temp[all_features].to_numpy()
    output_array = df_temp[output].to_numpy()
    
    return feature_matrix, output_array

example_features = ['sqft_living']
example_output = 'price'
(example_matrix, example_output_array) = get_numpy_data(train_data, example_features, example_output)

print("Task 1 checking 1:")
print(f"Перший рядок матриці ознак: {example_matrix[0]}")

Task 1 checking 1:
Перший рядок матриці ознак: [1.00e+00 4.01e+03]


In [3]:
def predict_output(feature_matrix, weights):
    predictions = np.dot(feature_matrix, weights)
    return predictions

test_weights = np.array([1.0, 1.0])

test_predictions = predict_output(example_matrix, test_weights)

print(f"Прогноз для першого будинку у вибірці: {test_predictions[0]}")

Прогноз для першого будинку у вибірці: 4011.0


In [4]:
def feature_derivative(errors, feature):
   
    derivative = 2 * np.dot(errors, feature)
    return derivative

(example_matrix, example_output) = get_numpy_data(train_data, ['sqft_living'], 'price')
zero_weights = np.array([0., 0.])
predictions = predict_output(example_matrix, zero_weights)
errors = predictions - example_output

derivative = feature_derivative(errors, example_matrix[:,0])
expected_derivative = -2 * np.sum(example_output)

print(f"Перевірка Завдання 3:")
print(f"Розрахована похідна: {derivative}")
print(f"Очікувана похідна:  {expected_derivative}")

Перевірка Завдання 3:
Розрахована похідна: -4646261332.0
Очікувана похідна:  -4646261332.0


In [5]:
def regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance):
    weights = np.array(initial_weights)
    converged = False
    
    while not converged:
        predictions = predict_output(feature_matrix, weights)
        errors = predictions - output
        
        gradient_sum_sq = 0
        
        for i in range(len(weights)):
            derivative = feature_derivative(errors, feature_matrix[:, i])
            gradient_sum_sq += derivative**2
            
            weights[i] = weights[i] - (step_size * derivative)
        
        l2_norm = np.sqrt(gradient_sum_sq)
        if l2_norm < tolerance:
            converged = True
            
    return weights

simple_features = ['sqft_living']
my_output = 'price'
(train_matrix, train_output) = get_numpy_data(train_data, simple_features, my_output)

initial_weights = np.array([-47000., 1.])
step_size = 7e-12
tolerance = 2.5e7

final_weights = regression_gradient_descent(train_matrix, train_output, initial_weights, step_size, tolerance)

(test_matrix, test_output) = get_numpy_data(test_data, simple_features, my_output)
test_predictions = predict_output(test_matrix, final_weights)

first_house_prediction = test_predictions[0]
rss_test = np.sum((test_predictions - test_output)**2)

print(f"Фінальні ваги: {final_weights}")
print(f"Передбачувана ціна 1-го будинку в тесті: ${first_house_prediction:,.2f}")
print(f"RSS на тестовій вибірці: {rss_test:.2e}")

Фінальні ваги: [-46999.88018846    280.51005013]
Передбачувана ціна 1-го будинку в тесті: $354,129.49
RSS на тестовій вибірці: 1.20e+15
